# Ghi log thay đổi nhiều tab Google Sheet → Supabase (Google Colab)

**Không cần Google Cloud Console / Service Account** – đăng nhập bằng chính tài khoản Google của bạn.

**Các tab được theo dõi:** Du Lịch, Sức khỏe, Oto, BHYT/BHXH, Thông tin cấp bảo hiểm xe máy, Bảo hiểm rủi ro.
Tab **template** (và mọi tab không nằm trong danh sách) **không** được ghi log.

**Chuẩn bị 1 lần:**
1. Bấm biểu tượng 🔑 **Secrets** ở thanh bên trái Colab → **Add new secret**:
   - Name: `SUPABASE_DB_URL`
   - Value: chuỗi kết nối Supabase (*Project Settings → Database → Connection string → **Session pooler***, thay `[YOUR-PASSWORD]` bằng mật khẩu DB).
   - Bật công tắc **Notebook access**.
2. Tài khoản Google bạn đăng nhập ở bước 2 phải mở được Google Sheet.

**Cách dùng:** chạy lần lượt các ô từ trên xuống.
- Ô **"Kiểm tra 1 lần"**: chạy thủ công mỗi khi muốn quét thay đổi.
- Ô **"Chạy tự động"**: quét liên tục khi tab Colab còn mở.

Mỗi tab có snapshot riêng (theo `gid`). Lần quét đầu của mỗi tab chỉ lưu snapshot làm mốc (không ghi log). Snapshot lưu **trên Supabase**, nên Colab bị reset cũng không mất.
Tất cả các tab được đọc bằng **1 lệnh API duy nhất** mỗi lần quét → không lo vượt quota Google.

## 1. Cài thư viện & cấu hình

In [ ]:
!pip install -q gspread psycopg2-binary

SPREADSHEET_ID = "1qc_QhrvpoLLp6w9RkGBEkm8qBO49GJE8oMlwkCdJOsk"

# Danh sách tab cần theo dõi (so khớp theo tên, không phân biệt hoa/thường, bỏ khoảng trắng thừa)
SHEET_NAMES = [
    "Du Lịch",
    "Sức khỏe",
    "Oto",
    "BHYT/BHXH",
    "Thông tin cấp bảo hiểm xe máy",
    "Bảo hiểm rủi ro",
]
# Chốt chặn an toàn: tab nào có tên chứa các từ này sẽ luôn bị bỏ qua
EXCLUDE_KEYWORDS = ["template", "templete"]

LOG_TABLE      = "sheet_change_log"   # bảng ghi log
SNAPSHOT_TABLE = "sheet_snapshot"     # bảng lưu trạng thái lần quét trước
POLL_INTERVAL  = 30                   # giây (dùng cho chế độ tự động)

from google.colab import userdata
SUPABASE_DB_URL = userdata.get("SUPABASE_DB_URL")
print("Đã đọc cấu hình ✔")

## 2. Đăng nhập Google & xác định các tab cần theo dõi

In [ ]:
from google.colab import auth
import google.auth, gspread, unicodedata

auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)
sh = gc.open_by_key(SPREADSHEET_ID)

def _norm(s):
    # NFC để tên tiếng Việt gõ kiểu tổ hợp / dựng sẵn vẫn khớp nhau
    return " ".join(unicodedata.normalize("NFC", s).split()).casefold()

def resolve_worksheets():
    all_ws = sh.worksheets()
    by_name = {_norm(w.title): w for w in all_ws}
    watched, missing = [], []
    for name in SHEET_NAMES:
        w = by_name.get(_norm(name))
        if w is None:
            missing.append(name)
        elif any(k in _norm(w.title) for k in EXCLUDE_KEYWORDS):
            print(f"  Bỏ qua tab template: '{w.title}'")
        else:
            watched.append(w)
    return all_ws, watched, missing

all_ws, WATCHED, missing = resolve_worksheets()

print(f"File: '{sh.title}'")
print(f"Đang theo dõi {len(WATCHED)} tab:")
for w in WATCHED:
    print(f"  ✔ {w.title}  (gid={w.id})")
skipped = [w.title for w in all_ws if w.id not in {x.id for x in WATCHED}]
if skipped:
    print("Không theo dõi:", ", ".join(skipped))
if missing:
    print("\n⚠ Không tìm thấy tab:", missing)
    print("  Hãy sửa SHEET_NAMES ở ô 1 cho đúng với tên tab thật ở trên rồi chạy lại ô này.")

## 3. Kết nối Supabase & tạo bảng (tự động nếu chưa có)

In [ ]:
import psycopg2
from psycopg2.extras import Json, execute_values

if not SUPABASE_DB_URL:
    raise RuntimeError("Chưa có secret SUPABASE_DB_URL (hoặc chưa bật Notebook access) – xem hướng dẫn ở đầu notebook.")

def db_connect():
    """Mở kết nối tới Supabase (Session pooler), có keepalive để hạn chế bị ngắt khi chạy lâu."""
    c = psycopg2.connect(
        SUPABASE_DB_URL,
        sslmode="require",
        connect_timeout=15,
        keepalives=1, keepalives_idle=30, keepalives_interval=10, keepalives_count=5,
        application_name="sheet_logger_colab",
    )
    c.autocommit = False
    return c

DDL = f"""
create table if not exists public.{LOG_TABLE} (
    id             bigserial primary key,
    batch_id       uuid        not null,
    spreadsheet_id text        not null,
    sheet_gid      bigint      not null,
    sheet_name     text,
    cell           text        not null,
    row_num        integer     not null,
    col_num        integer     not null,
    column_header  text,
    change_type    text        not null check (change_type in ('INSERT','UPDATE','DELETE')),
    old_value      text,
    new_value      text,
    detected_at    timestamptz not null default now()
);
create index if not exists {LOG_TABLE}_sheet_time_idx
    on public.{LOG_TABLE} (spreadsheet_id, sheet_gid, detected_at desc);
create index if not exists {LOG_TABLE}_batch_idx
    on public.{LOG_TABLE} (batch_id);

create table if not exists public.{SNAPSHOT_TABLE} (
    spreadsheet_id text        not null,
    sheet_gid      bigint      not null,
    data           jsonb       not null,
    fingerprint    text        not null,
    updated_at     timestamptz not null default now(),
    primary key (spreadsheet_id, sheet_gid)
);

-- Bật RLS để bảng không bị lộ qua API công khai (anon key) của Supabase.
-- Kết nối trực tiếp bằng user postgres (chủ bảng) vẫn đọc/ghi bình thường.
alter table public.{LOG_TABLE}      enable row level security;
alter table public.{SNAPSHOT_TABLE} enable row level security;
"""

conn = db_connect()
with conn.cursor() as cur:
    cur.execute(DDL)
    cur.execute("select current_user, version()")
    user, ver = cur.fetchone()
conn.commit()

print(f"Đã kết nối Supabase với user '{user}' ✔")
print(f"  {ver.split(',')[0]}")
print(f"Bảng sẵn sàng: public.{LOG_TABLE}, public.{SNAPSHOT_TABLE} ✔")

## 4. Hàm so sánh & ghi log

In [ ]:
import hashlib, json, uuid
from gspread.utils import rowcol_to_a1

def fingerprint(values):
    return hashlib.sha256(json.dumps(values, ensure_ascii=False).encode()).hexdigest()

def load_snapshot(conn, gid):
    with conn.cursor() as cur:
        cur.execute(f"select data, fingerprint from public.{SNAPSHOT_TABLE} "
                    "where spreadsheet_id=%s and sheet_gid=%s", (SPREADSHEET_ID, gid))
        row = cur.fetchone()
    return (row[0], row[1]) if row else (None, None)

def save_snapshot(cur, gid, values, fp):
    cur.execute(f"""
        insert into public.{SNAPSHOT_TABLE} (spreadsheet_id, sheet_gid, data, fingerprint, updated_at)
        values (%s, %s, %s, %s, now())
        on conflict (spreadsheet_id, sheet_gid)
        do update set data = excluded.data, fingerprint = excluded.fingerprint, updated_at = now()
    """, (SPREADSHEET_ID, gid, Json(values), fp))

def diff_values(old, new):
    changes = []
    for r in range(max(len(old), len(new))):
        o_row = old[r] if r < len(old) else []
        n_row = new[r] if r < len(new) else []
        for c in range(max(len(o_row), len(n_row))):
            ov = o_row[c] if c < len(o_row) else ""
            nv = n_row[c] if c < len(n_row) else ""
            if ov == nv:
                continue
            ctype = "INSERT" if ov == "" else "DELETE" if nv == "" else "UPDATE"
            changes.append((r + 1, c + 1, ov or None, nv or None, ctype))
    return changes

def _a1_sheet(title):
    # Đặt tên tab trong nháy đơn (cần cho tên có dấu cách, '/', tiếng Việt...)
    return "'" + title.replace("'", "''") + "'"

def fetch_all_sheets():
    """Đọc toàn bộ các tab đang theo dõi bằng 1 lệnh API. Trả về {gid: values}."""
    resp = sh.values_batch_get([_a1_sheet(w.title) for w in WATCHED])
    result = {}
    for w, vr in zip(WATCHED, resp.get("valueRanges", [])):
        vals = vr.get("values", [])
        width = max((len(r) for r in vals), default=0)
        result[w.id] = [r + [""] * (width - len(r)) for r in vals]   # đệm cho đều cột
    return result

def check_sheet(w, current):
    """So sánh 1 tab với snapshot, ghi log + cập nhật snapshot trong cùng transaction."""
    fp = fingerprint(current)
    old, old_fp = load_snapshot(conn, w.id)
    tag = f"[{w.title}]"

    with conn.cursor() as cur:
        if old is None:
            save_snapshot(cur, w.id, current, fp)
            conn.commit()
            print(f"{tag} Lần đầu: đã lưu snapshot mốc ({len(current)} dòng), chưa ghi log.")
            return 0
        if fp == old_fp:
            conn.rollback()
            print(f"{tag} Không có thay đổi.")
            return 0

        changes = diff_values(old, current)
        header = current[0] if current else []
        batch_id = str(uuid.uuid4())
        rows = [
            (batch_id, SPREADSHEET_ID, w.id, w.title, rowcol_to_a1(r, c), r, c,
             header[c - 1] if (r > 1 and c - 1 < len(header)) else None, t, ov, nv)
            for r, c, ov, nv, t in changes
        ]
        if rows:
            execute_values(cur, f"""
                insert into public.{LOG_TABLE}
                (batch_id, spreadsheet_id, sheet_gid, sheet_name, cell, row_num, col_num,
                 column_header, change_type, old_value, new_value)
                values %s""", rows, page_size=500)
        save_snapshot(cur, w.id, current, fp)
    conn.commit()

    print(f"{tag} Phát hiện {len(rows)} ô thay đổi → đã đẩy lên Supabase:")
    for r in rows[:20]:
        print(f"    {r[4]:>6} | {r[8]:<6} | {r[9]!r} → {r[10]!r}")
    if len(rows) > 20:
        print(f"    ... và {len(rows) - 20} ô khác")
    return len(rows)

def check_once():
    """Quét tất cả các tab 1 lần. Trả về tổng số ô thay đổi."""
    global conn, WATCHED
    if conn.closed:
        conn = db_connect()
    try:
        data = fetch_all_sheets()
    except gspread.exceptions.APIError as e:
        # Có thể do tab vừa bị đổi tên → dò lại danh sách tab rồi thử lại 1 lần
        if "Unable to parse range" not in str(e):
            raise
        print("Tên tab có thể đã đổi, đang dò lại...")
        _, WATCHED, miss = resolve_worksheets()
        if miss:
            print("⚠ Không tìm thấy tab:", miss)
        data = fetch_all_sheets()

    total = 0
    for w in WATCHED:
        total += check_sheet(w, data.get(w.id, []))
    return total

print("Đã nạp hàm ✔")

## 5A. Kiểm tra 1 lần (chạy thủ công)
Chạy lại ô này bất cứ khi nào muốn quét thay đổi.

In [ ]:
check_once()

## 5B. Chạy tự động (tuỳ chọn)
Quét mỗi `POLL_INTERVAL` giây. Bấm nút ⏹ **Stop** để dừng. Colab sẽ ngắt khi tab bị đóng hoặc idle lâu.

In [ ]:
import time, datetime
from IPython.display import clear_output

MAX_BACKOFF   = 600   # giây – thời gian chờ tối đa khi gặp lỗi liên tiếp
CLEAR_EVERY   = 100   # xoá bớt output sau mỗi N vòng để Colab không bị nặng

def _reset_conn():
    global conn
    try:
        conn.close()   # check_once() sẽ tự kết nối lại vì conn.closed = True
    except Exception:
        pass

print(f"▶ Bắt đầu quét tự động mỗi {POLL_INTERVAL}s – bấm ⏹ Stop để dừng.")
loop, errors, total_changes = 0, 0, 0
try:
    while True:
        loop += 1
        if loop % CLEAR_EVERY == 0:
            clear_output(wait=True)
            print(f"(Đã xoá output cũ) Tổng số ô thay đổi từ lúc chạy: {total_changes}")

        now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n─── Vòng {loop} · {now} ───")
        try:
            n = check_once()
            total_changes += n
            errors = 0
        except (psycopg2.OperationalError, psycopg2.InterfaceError) as e:
            errors += 1
            print(f"⚠ Mất kết nối Supabase: {str(e).strip()} → sẽ kết nối lại ở vòng sau.")
            _reset_conn()
        except gspread.exceptions.APIError as e:
            errors += 1
            print(f"⚠ Lỗi Google API (có thể vượt quota/mạng chập chờn): {e}")
        except Exception as e:
            errors += 1
            print(f"⚠ Lỗi không mong muốn: {type(e).__name__}: {e}")
            try:
                conn.rollback()     # tránh để transaction ở trạng thái lỗi
            except Exception:
                _reset_conn()

        wait = POLL_INTERVAL if errors == 0 else min(POLL_INTERVAL * 2 ** errors, MAX_BACKOFF)
        if errors:
            print(f"  Lỗi liên tiếp: {errors} → chờ {wait}s rồi thử lại.")
        time.sleep(wait)
except KeyboardInterrupt:
    print(f"\n⏹ Đã dừng sau {loop} vòng. Tổng số ô thay đổi đã ghi: {total_changes}")

## 6. Xem log gần nhất

In [ ]:
import pandas as pd, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
if conn.closed:
    conn = db_connect()

gids = [w.id for w in WATCHED]

# Tổng hợp số thay đổi theo từng tab
display(pd.read_sql(f"""
    select sheet_name, count(*) as so_o_thay_doi, max(detected_at) as lan_cuoi
    from public.{LOG_TABLE}
    where spreadsheet_id = %s and sheet_gid = any(%s)
    group by sheet_name order by lan_cuoi desc
""", conn, params=(SPREADSHEET_ID, gids)))

# 50 log gần nhất (đổi FILTER_SHEET thành tên tab để lọc riêng, ví dụ "Oto")
FILTER_SHEET = None
q = f"""select detected_at, sheet_name, cell, column_header, change_type, old_value, new_value
        from public.{LOG_TABLE}
        where spreadsheet_id = %s and sheet_gid = any(%s)"""
params = [SPREADSHEET_ID, gids]
if FILTER_SHEET:
    q += " and sheet_name = %s"; params.append(FILTER_SHEET)
q += " order by id desc limit 50"
pd.read_sql(q, conn, params=params)